In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [2]:
class SimpleNetWithInhibition(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_inhibitory, dropout_rate=0.5):
        super(SimpleNetWithInhibition, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_rate)
        self.num_inhibitory = num_inhibitory
        if self.num_inhibitory > 0:
            self.W_inh = nn.Parameter(torch.randn(num_inhibitory, hidden_size) * 0.01)

    def forward(self, x):
        x = x.view(-1, 28*28)  # Flatten the input
        h = torch.relu(self.fc1(x))
        h = self.dropout(h)  # Apply dropout
        if self.num_inhibitory > 0:
            inhibition = torch.matmul(h[:, :self.num_inhibitory], self.W_inh)
            h = torch.cat((h[:, :self.num_inhibitory] - inhibition, h[:, self.num_inhibitory:]), dim=1)
        y = self.fc2(h)
        return y
 
# Initialize networks
input_size = 28 * 28
hidden_size = 128
output_size = 10

In [3]:
def train_network(network, train_loader, test_loader, dropout_rate, num_epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(network.parameters(), lr=lr)
    
    for epoch in range(num_epochs):
        network.train()
        for data, target in train_loader:
            data = nn.functional.dropout(data, p=dropout_rate, training=True)
            optimizer.zero_grad()
            output = network(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    # Evaluate the network
    network.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data = nn.functional.dropout(data, p=dropout_rate, training=False)
            output = network(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [4]:
dropout_rates = np.linspace(0, 0.9, 10)
ratios = [1, 2, 5, 10, 20, 50]
num_epochs = 50

results = {}

for dropout_rate in dropout_rates:
    for ratio in ratios:
        num_inhibitory = hidden_size // ratio
        num_excitatory = hidden_size - num_inhibitory
        
        net_with_inh = SimpleNetWithInhibition(input_size, hidden_size, output_size, num_inhibitory, dropout_rate)
        net_without_inh = SimpleNetWithInhibition(input_size, hidden_size, output_size, 0, dropout_rate)
        
        print(f"Dropout rate: {dropout_rate}, Ratio: {ratio}, With inhibition")
        acc_with_inh = train_network(net_with_inh, train_loader, test_loader, dropout_rate, num_epochs)
        
        print(f"Dropout rate: {dropout_rate}, Ratio: {ratio}, Without inhibition")
        acc_without_inh = train_network(net_without_inh, train_loader, test_loader, dropout_rate, num_epochs)
        
        results[(dropout_rate, ratio, 'with_inh')] = acc_with_inh
        results[(dropout_rate, ratio, 'without_inh')] = acc_without_inh

# Print results
for key in sorted(results.keys()):
    dropout_rate, ratio, inhibition_status = key
    accuracy = results[key]
    print(f"Dropout rate: {dropout_rate}, Ratio: {ratio}, {inhibition_status}: {accuracy:.2f}%")

Dropout rate: 0.0, Ratio: 1, With inhibition
Dropout rate: 0.0, Ratio: 1, Without inhibition
Dropout rate: 0.0, Ratio: 2, With inhibition


RuntimeError: The size of tensor a (64) must match the size of tensor b (128) at non-singleton dimension 1

In [ ]:
import matplotlib.pyplot as plt

plt.plot(results_with_inh)
plt.plot(results_without_inh)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load results
results = np.load('results.npy', allow_pickle=True).item()

dropout_rates = np.linspace(0, 0.9, 10)
ratios = [1, 2, 5, 10, 20, 50]

# Accuracy vs. Dropout Rate Plot
plt.figure(figsize=(12, 6))

for ratio in ratios:
    acc_with_inh = [results[(dropout_rate, ratio, 'with_inh')] for dropout_rate in dropout_rates]
    acc_without_inh = [results[(dropout_rate, ratio, 'without_inh')] for dropout_rate in dropout_rates]
    plt.plot(dropout_rates, acc_with_inh, label=f'With Inhibition, Ratio={ratio}')
    plt.plot(dropout_rates, acc_without_inh, linestyle='dashed', label=f'Without Inhibition, Ratio={ratio}')

plt.scatter([0], [results[(0, ratio, 'with_inh')] for ratio in ratios], color='blue', s=100, zorder=5, label='No Dropout (With Inhibition)')
plt.scatter([0], [results[(0, ratio, 'without_inh')] for ratio in ratios], color='red', s=100, zorder=5, label='No Dropout (Without Inhibition)')

plt.xlabel('Dropout Rate')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Dropout Rate')
plt.legend()
plt.grid(True)
plt.show()

# Accuracy vs. Ratio Plot
plt.figure(figsize=(12, 6))

for dropout_rate in dropout_rates:
    acc_with_inh = [results[(dropout_rate, ratio, 'with_inh')] for ratio in ratios]
    acc_without_inh = [results[(dropout_rate, ratio, 'without_inh')] for ratio in ratios]
    plt.plot(ratios, acc_with_inh, label=f'With Inhibition, Dropout Rate={dropout_rate}')
    plt.plot(ratios, acc_without_inh, linestyle='dashed', label=f'Without Inhibition, Dropout Rate={dropout_rate}')

plt.scatter([0], [results[(dropout_rate, 1, 'with_inh')] for dropout_rate in dropout_rates], color='blue', s=100, zorder=5, label='No Inhibition (With Dropout)')
plt.scatter([0], [results[(dropout_rate, 1, 'without_inh')] for dropout_rate in dropout_rates], color='red', s=100, zorder=5, label='No Inhibition (Without Dropout)')

plt.xlabel('Ratio of Excitatory to Inhibitory Neurons')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Ratio of Excitatory to Inhibitory Neurons')
plt.legend()
plt.grid(True)
plt.xscale('log')
plt.show()